In [1]:
# Install
!pip install -q transformers torchaudio soundfile

# Imports
import torch
import soundfile as sf
from transformers import (
    SpeechT5Processor,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to(device)
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

# Random speaker embedding (سريع جدًا)
speaker_embedding = torch.randn(1, 512).to(device)

# Text
text = "Hello this is a quick text to speech test."

inputs = processor(text=text, return_tensors="pt").to(device)

# Generate speech
speech = model.generate_speech(
    inputs["input_ids"],
    speaker_embedding,
    vocoder=vocoder
)

# Save file
sf.write("quick_tts.wav", speech.cpu().numpy(), 16000)

print("Done ✅ Audio saved as quick_tts.wav")

preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/50.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/50.6M [00:00<?, ?B/s]

Done ✅ Audio saved as quick_tts.wav


In [2]:
# =====================================
# INSTALL
# =====================================
!pip install -q transformers datasets accelerate torchaudio soundfile

# =====================================
# IMPORTS
# =====================================
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    SpeechT5Processor,
    SpeechT5ForTextToSpeech,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================
# LOAD DATA
# =====================================
dataset = load_dataset(
    "facebook/voxpopuli",
    "it",
    split="train[:5]"
)

dataset = dataset.train_test_split(test_size=0.2)

# =====================================
# LOAD MODEL
# =====================================
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")

model = SpeechT5ForTextToSpeech.from_pretrained(
    "microsoft/speecht5_tts"
).to(device)

# 🔥 أهم سطرين لحل الخطأ
model.config.use_guided_attention_loss = False
model.config.use_masking = False

# =====================================
# PREPROCESS
# =====================================
def prepare_dataset(example):

    audio = example["audio"]

    processed = processor(
        text=example["normalized_text"],
        audio_target=audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_attention_mask=False
    )

    processed["input_ids"] = torch.tensor(processed["input_ids"])
    processed["labels"] = torch.tensor(processed["labels"][0])
    processed["speaker_embeddings"] = torch.randn(512)

    return processed

dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names
)

dataset.set_format("torch")

# =====================================
# DATA COLLATOR
# =====================================
def data_collator(features):

    input_ids = torch.nn.utils.rnn.pad_sequence(
        [f["input_ids"] for f in features],
        batch_first=True,
        padding_value=processor.tokenizer.pad_token_id
    )

    labels = torch.nn.utils.rnn.pad_sequence(
        [f["labels"] for f in features],
        batch_first=True
    )

    speaker_embeddings = torch.stack(
        [f["speaker_embeddings"] for f in features]
    )

    return {
        "input_ids": input_ids,
        "labels": labels,
        "speaker_embeddings": speaker_embeddings,
    }

# =====================================
# TRAINING ARGS
# =====================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./mini_tts",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    max_steps=10,
    logging_steps=2,
    learning_rate=1e-5,
    report_to="none"
)

# =====================================
# TRAINER
# =====================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator
)

# =====================================
# TRAIN
# =====================================
trainer.train()

print("Mini Fine-Tuning Done ✅")

README.md: 0.00B [00:00, ?B/s]

it/train-00000-of-00005.parquet:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

it/train-00001-of-00005.parquet:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

it/train-00002-of-00005.parquet:   0%|          | 0.00/3.08G [00:00<?, ?B/s]

it/train-00003-of-00005.parquet:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

...

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
2,2.846626
4,3.242793
6,1.879192
8,2.611431
10,2.313748


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Mini Fine-Tuning Done ✅
